##  Text Representation: a set of numbers that represent what the text means and capture the semantics of the text. These numbers are called text embeddings.

In [1]:
import pandas as pd
import numpy as np
import altair as alt
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans

In [2]:
!pip install cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 23.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [cohere]


In [3]:
import cohere
co = cohere.ClientV2("") # Get your free API key: https://dashboard.cohere.com/api-keys

In [4]:
# Load the dataset to a dataframe
df_orig = pd.read_csv('https://raw.githubusercontent.com/cohere-ai/notebooks/main/notebooks/data/atis_intents_train.csv', names=['intent','query'])

# Take a small sample for illustration purposes
sample_classes = ['atis_airfare', 'atis_airline', 'atis_ground_service']
df = df_orig.sample(frac=0.1, random_state=30)
df = df[df.intent.isin(sample_classes)]
df_orig = df_orig.drop(df.index)
df.reset_index(drop=True,inplace=True)

# Remove unnecessary column
intents = df['intent'] #save for a later need
df.drop(columns=['intent'], inplace=True)

In [5]:
def get_embeddings(texts, model="embed-v4.0", input_type="search_document"):
    output = co.embed(
        texts=texts,
        model=model,
        input_type=input_type,
        embedding_types=["float"]
    )
    return output.embeddings.float

df['query_embeds'] = get_embeddings(df['query'].tolist())

In [6]:
def get_pc(arr, n):
    pca = PCA(n_components=n)
    embeds_transform = pca.fit_transform(arr)
    return embeds_transform

# Reduce embeddings to 10 principal components to aid visualization
embeds = np.array(df['query_embeds'].tolist())
embeds_pc = get_pc(embeds, 10)

### Visualizing PCA Embeddings with a Heatmap

We will visualize the first 10 principal components for the first 10 data points using `altair`. This provides an easy way to see how the reduced features represent each query.

In [7]:
# Prepare the data for the heatmap of the first 10 documents and their 10 principal components
n_queries = 10
n_pcs = 10

# Create a list of records for the dataframe
heatmap_data = []
for i in range(n_queries):
    query_text = df['query'].iloc[i]
    # Truncate text for cleaner visualization labels
    short_query = (query_text[:40] + '...') if len(query_text) > 40 else query_text
    for j in range(n_pcs):
        heatmap_data.append({
            'Query Index': f"{i}: {short_query}",
            'Principal Component': f"PC {j+1}",
            'Value': embeds_pc[i, j]
        })

df_heatmap = pd.DataFrame(heatmap_data)

# Create the Altair heatmap
heatmap = alt.Chart(df_heatmap).mark_rect().encode(
    x=alt.X('Principal Component:O', sort=None, title='Principal Components'),
    y=alt.Y('Query Index:O', sort=None, title='Queries'),
    color=alt.Color('Value:Q', scale=alt.Scale(scheme='viridis'), title='PC Value'),
    tooltip=['Query Index', 'Principal Component', 'Value']
).properties(
    width=500,
    height=300,
    title='PCA Embeddings Heatmap (First 10 Queries)'
)

heatmap

alt.Chart(...)